# OrdinalCLIP Few-Shot Simulation - DeepDRiD

This notebook perfectly mimics the original few-shot simulation methodology used by the `OrdinalCLIP` repository for Morph, but applied to the DeepDRiD dataset.

It dynamically samples $K$ shots per class using `generate_fewshot_dataset` and trains the model.

In [ ]:
import os, sys, warnings
from pathlib import Path
import pandas as pd
import torch
import pytorch_lightning as pl
from omegaconf import OmegaConf

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

# Add original OrdinalCLIP repo to path
ORDINALCLIP_REPO = ROOT / 'repos' / 'OrdinalCLIP'
sys.path.insert(0, str(ORDINALCLIP_REPO))

# Import Original OrdinalCLIP runner & dataset modules
from ordinalclip.runner.data import RegressionDataModule
from ordinalclip.runner.runner import Runner
from pytorch_lightning.callbacks import ModelCheckpoint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root   : {ROOT}')
print(f'OrdinalCLIP    : {ORDINALCLIP_REPO}')
print(f'Device         : {DEVICE}')

## Simulation Loop Configuration

In [ ]:

K_SHOTS = [2, 4, 8, 16]
SEEDS   = [1, 2, 3]

RESULTS_DIR = ROOT / "results" / "OrdinalCLIP_Original" / "DeepDRiD"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Dataset paths
DEEPDR_ROOT = Path("G:/Datasets/DR/DeepDR/DeepDRiD-master/regular_fundus_images")
SPLITS_DIR = ROOT / "data" / "DeepDRiD" / "ordinalclip_splits"

# Base configuration
base_cfg = OmegaConf.load(str(ORDINALCLIP_REPO / "configs" / "default.yaml"))

# Override Dataset config
base_cfg.data_cfg.train_images_root = str(DEEPDR_ROOT)
base_cfg.data_cfg.val_images_root   = str(DEEPDR_ROOT)
base_cfg.data_cfg.test_images_root  = str(DEEPDR_ROOT)

base_cfg.data_cfg.train_data_file   = str(SPLITS_DIR / "train.txt")
base_cfg.data_cfg.val_data_file     = str(SPLITS_DIR / "val.txt")
base_cfg.data_cfg.test_data_file    = str(SPLITS_DIR / "val.txt")

# We only have 5 ranks for DeepDRiD
base_cfg.runner_cfg.model_cfg.prompt_learner_cfg.num_ranks = 5

base_cfg.trainer_cfg.max_epochs = 50
base_cfg.trainer_cfg.devices = 1
base_cfg.trainer_cfg.accelerator = 'auto'
try:
    base_cfg.trainer_cfg.precision = 'bf16-mixed' if torch.cuda.is_bf16_supported() else '32-true'
except:
    base_cfg.trainer_cfg.precision = '32-true'


## Run Grid

In [ ]:

for k in K_SHOTS:
    for seed in SEEDS:
        print(f"\n{'='*50}\nStarting K={k}, Seed={seed}\n{'='*50}")
        
        cfg = OmegaConf.create(OmegaConf.to_container(base_cfg, resolve=True))
        
        # Override few-shot config & seed
        cfg.data_cfg.few_shot.num_shots = k
        cfg.runner_cfg.seed = seed
        
        # Output directory for this specific run
        run_dir = RESULTS_DIR / f"{k}shot_s{seed}"
        cfg.runner_cfg.output_dir = str(run_dir)
        
        pl.seed_everything(seed, workers=True)
        
        # Check if already completed
        if (run_dir / "test_results.yaml").exists():
            print("Already completed, skipping.")
            continue
            
        datamodule = RegressionDataModule(**OmegaConf.to_container(cfg.data_cfg))
        
        callbacks = [
            ModelCheckpoint(
                monitor="val_mae_max_metric",
                dirpath=str(run_dir / "ckpts"),
                filename="{epoch:02d}",
                save_last=True,
                save_top_k=1,
                mode="min",
                save_weights_only=True,
            )
        ]
        
        trainer = pl.Trainer(
            callbacks=callbacks,
            logger=False,
            enable_checkpointing=True,
            enable_progress_bar=True,
            **OmegaConf.to_container(cfg.trainer_cfg)
        )
        
        runner = Runner(**OmegaConf.to_container(cfg.runner_cfg))
        
        trainer.fit(model=runner, datamodule=datamodule)
        
        # Test
        print("Testing...")
        best_ckpt = callbacks[0].best_model_path
        if best_ckpt:
            import omegaconf
            try: torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])
            except: pass
            
            # The original runner does not load weights automatically before testing,
            # we do it manually to test the best epoch
            runner = Runner.load_from_checkpoint(best_ckpt, **OmegaConf.to_container(cfg.runner_cfg), weights_only=False)
            
        trainer.test(model=runner, datamodule=datamodule)
        
        # Save metrics
        OmegaConf.save(OmegaConf.create(runner.test_metrics), str(run_dir / "test_results.yaml"))


In [ ]:
print("Grid complete!")